# 🪙 Neiro-Quant — Market Microstructure & GEX Terminal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DaniNguyen2509/Neiro-Quant/blob/main/notebooks/Neiro_Quant.ipynb)

> **Hệ thống Định lượng Vị thế Nhà tạo lập (Dealer Positioning), Gamma Exposure (GEX) & Tích hợp Trí tuệ Nhân tạo DeepSeek.**  
> **Tác chiến trước 3 phiên:** Phiên Á (06:30) • Phiên Âu (13:30) • Phiên Mỹ (19:00 VN)  
> **Tài sản:** Gold Futures (`GC` CME) & Gold Spot (`XAUUSD`)


### ⚙️ Bước 1: Cài đặt thư viện cần thiết

In [ ]:
!pip install -q plotly pandas numpy openai requests

### 🔑 Bước 2: Cấu hình Khóa API DeepSeek & Thông số Phiên
*(Khóa API của anh đã được thiết lập sẵn, hoặc anh có thể lấy từ Colab Secrets)*

In [ ]:
import os
import getpass

# Tự động nạp hoặc dùng key của anh
try:
    from google.colab import userdata
    DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
except Exception:
    DEEPSEEK_API_KEY = None

if not DEEPSEEK_API_KEY:
    DEEPSEEK_API_KEY = 'sk-6859bd549ace4f77a7978daade9346f4'

# THIẾT LẬP THÔNG SỐ ĐẦU PHIÊN GIAO DỊCH
SESSION_NAME = 'PHIÊN ÂU (13:30 VN)'  # Thay bằng 'PHIÊN Á' hoặc 'PHIÊN MỸ'
FUTURES_PRICE = 2712.50              # Giá Vàng Futures GC trên Barchart
SPOT_PRICE = 2685.20                 # Giá Vàng Spot XAUUSD trên MT5 của anh

BASIS = round(FUTURES_PRICE - SPOT_PRICE, 2)
print(f'✅ Khóa DeepSeek API: {DEEPSEEK_API_KEY[:8]}...{DEEPSEEK_API_KEY[-4:]}')
print(f'📊 Phiên: {SESSION_NAME}')
print(f'⚖️ Futures: {FUTURES_PRICE} | Spot XAUUSD: {SPOT_PRICE} | Độ lệch Basis: {BASIS:+.2f} USD')


### 📥 Bước 3: Nạp dữ liệu Options Chain
- **Mặc định:** Sử dụng dữ liệu mẫu chuẩn của sàn CME trong kho repo `Neiro-Quant`.
- **Khi tác chiến:** Anh có thể tải file CSV tải từ Barchart lên hoặc dán thẳng vào ô bên dưới.

In [ ]:
import pandas as pd
import numpy as np
import os
import io

# ==============================================================================
# HÀM TỰ ĐỘNG QUÉT & NẠP DỮ LIỆU TỪ THƯ MỤC DATA/ (ƯU TIÊN FILE MỚI NHẤT)
# ==============================================================================
def load_and_parse_options_data(filepath_or_buffer=None):
    if filepath_or_buffer is not None:
        raw_source = filepath_or_buffer
    else:
        # 1. Quét ưu tiên thư mục data/
        candidates = []
        search_dirs = ['data', '../data', '.', 'docs/tailieudacta']
        
        for d in search_dirs:
            if os.path.exists(d):
                for f in os.listdir(d):
                    if f.endswith('.csv') and ('option' in f.lower() or 'gcz' in f.lower() or 'xau' in f.lower()):
                        full_p = os.path.join(d, f)
                        candidates.append((os.path.getmtime(full_p), full_p))
        
        # Sắp xếp lấy file mới nhất (most recently modified)
        if len(candidates) > 0:
            candidates.sort(reverse=True) # Mới nhất lên đầu
            raw_source = candidates[0][1]
            print(f'📂 Đang nạp file CSV mới nhất: "{raw_source}"')
        else:
            raw_source = None

    if raw_source is not None:
        try:
            with open(raw_source, 'r', encoding='utf-8', errors='ignore') as f:
                lines = [line.strip() for line in f if line.strip() and not line.startswith('"Downloaded from')]
            df_raw = pd.read_csv(io.StringIO('\n'.join(lines)))
            
            def clean_k(val):
                if pd.isna(val): return np.nan
                s = str(val).replace(',', '').replace('"', '').replace('C', '').replace('P', '').strip()
                try: return float(s)
                except: return np.nan

            df_raw['strike_clean'] = df_raw['Strike'].apply(clean_k)
            
            # Tách Calls và Puts (Định dạng Stacked chuẩn của Barchart)
            calls = df_raw[df_raw['Type'] == 'Call'].copy()
            puts = df_raw[df_raw['Type'] == 'Put'].copy()
            
            merged = pd.merge(calls, puts, on='strike_clean', suffixes=('_call', '_put')).sort_values('strike_clean').reset_index(drop=True)
            
            df_out = pd.DataFrame()
            df_out['strike'] = merged['strike_clean']
            df_out['callOpenInterest'] = pd.to_numeric(merged.get('Open Int_call', 0), errors='coerce').fillna(0)
            df_out['putOpenInterest']  = pd.to_numeric(merged.get('Open Int_put', 0), errors='coerce').fillna(0)
            df_out['callVolume'] = pd.to_numeric(merged.get('Volume_call', 0), errors='coerce').fillna(0)
            df_out['putVolume']  = pd.to_numeric(merged.get('Volume_put', 0), errors='coerce').fillna(0)
            
            c_bid = pd.to_numeric(merged.get('Bid_call', 0), errors='coerce').fillna(0)
            c_ask = pd.to_numeric(merged.get('Ask_call', 0), errors='coerce').fillna(0)
            c_latest = pd.to_numeric(merged.get('Latest_call', 0), errors='coerce').fillna(0)
            
            p_bid = pd.to_numeric(merged.get('Bid_put', 0), errors='coerce').fillna(0)
            p_ask = pd.to_numeric(merged.get('Ask_put', 0), errors='coerce').fillna(0)
            p_latest = pd.to_numeric(merged.get('Latest_put', 0), errors='coerce').fillna(0)
            
            df_out['callBid'] = np.where((c_bid > 0) & (c_ask > 0), c_bid, c_latest - 0.20)
            df_out['callAsk'] = np.where((c_bid > 0) & (c_ask > 0), c_ask, c_latest + 0.20)
            df_out['putBid']  = np.where((p_bid > 0) & (p_ask > 0), p_bid, p_latest - 0.20)
            df_out['putAsk']  = np.where((p_bid > 0) & (p_ask > 0), p_ask, p_latest + 0.20)
            
            df_out['callGamma'] = np.exp(-((FUTURES_PRICE - df_out['strike'])/45.0)**2) * 0.016
            df_out['putGamma']  = df_out['callGamma']
            
            print(f'✅ Nạp thành công {len(df_out)} Strikes từ {df_out["strike"].min():.0f} đến {df_out["strike"].max():.0f}!')
            return df_out
        except Exception as e:
            print('⚠️ Lỗi đọc file Barchart, chuyển sang dữ liệu mẫu chuẩn:', e)

    # Dự phòng mặc định nếu thư mục trống
    strikes = [2600, 2610, 2620, 2630, 2640, 2650, 2660, 2670, 2680, 2690, 2700, 2710, 2720, 2730, 2740, 2750, 2760, 2770, 2780, 2790, 2800]
    call_oi = [300, 320, 350, 380, 420, 450, 500, 600, 750, 1100, 1800, 1950, 2100, 1600, 1200, 4250, 800, 650, 500, 350, 200]
    put_oi  = [200, 250, 350, 500, 70, 85, 95, 4800, 2400, 1600, 1950, 1100, 800, 600, 450, 350, 250, 180, 120, 90, 50]
    
    rows = []
    for i, K in enumerate(strikes):
        c_mid = max(0.5, round((FUTURES_PRICE - K) if FUTURES_PRICE > K else 18.0 * np.exp(-abs(K - FUTURES_PRICE)/35.0), 2))
        p_mid = max(0.5, round((K - FUTURES_PRICE) if K > FUTURES_PRICE else 18.0 * np.exp(-abs(K - FUTURES_PRICE)/35.0), 2))
        rows.append({
            'strike': K, 'callBid': c_mid - 0.20, 'callAsk': c_mid + 0.20, 'callOpenInterest': call_oi[i],
            'putBid': p_mid - 0.20, 'putAsk': p_mid + 0.20, 'putOpenInterest': put_oi[i],
            'callGamma': 0.015, 'putGamma': 0.015
        })
    print('ℹ️ Thư mục data/ chưa có file mới, đang sử dụng bộ dữ liệu mẫu mặc định.')
    return pd.DataFrame(rows)

df = load_and_parse_options_data()
df.head(10)


### 🧮 Bước 4: Lõi tính toán Định lượng (Quant Engine Core)
Tính toán ma trận: Mid-Price, Trục ATM Synthetic Pivot, Dải băng $\pm 1\sigma$, GEX, Walls & Air Pockets.

In [ ]:
# 1. Mid-Price
df['callMid'] = (df['callBid'] + df['callAsk']) / 2.0
df['putMid'] = (df['putBid'] + df['putAsk']) / 2.0

# 2. K_ATM Pivot (Put-Call Parity)
df['diff'] = (df['callMid'] - df['putMid']).abs()
atm_idx = df['diff'].idxmin()
atm_row = df.loc[atm_idx]

K_ATM = atm_row['strike']
S_Implied = K_ATM + atm_row['callMid'] - atm_row['putMid']
Straddle = atm_row['callMid'] + atm_row['putMid']

Upper_1s = S_Implied + Straddle
Lower_1s = S_Implied - Straddle
Upper_085s = S_Implied + 0.85 * Straddle
Lower_085s = S_Implied - 0.85 * Straddle

# 3. GEX ($ per 1% move) - Multiplier = 100 troy oz
MULT = 100
df['callGEX'] = df['callGamma'] * df['callOpenInterest'] * MULT * (FUTURES_PRICE**2) * 0.01 / 1e6
df['putGEX']  = - (df['putGamma'] * df['putOpenInterest'] * MULT * (FUTURES_PRICE**2) * 0.01 / 1e6)
df['netGEX']  = df['callGEX'] + df['putGEX']

total_net_gex = df['netGEX'].sum()
gex_regime = 'LONG GAMMA (+GEX: Đè nén vol / Hấp thụ)' if total_net_gex > 0 else 'SHORT GAMMA (-GEX: Phóng dao / Trượt dốc)'

# 4. Major Walls
call_wall_row = df.loc[df['callOpenInterest'].idxmax()]
put_wall_row  = df.loc[df['putOpenInterest'].idxmax()]
abs_gamma_row = df.loc[df['netGEX'].abs().idxmax()]

Call_Wall_Futures = call_wall_row['strike']
Put_Wall_Futures  = put_wall_row['strike']
Abs_GEX_Futures   = abs_gamma_row['strike']

# 5. Air Pockets (Vùng rỗng thanh khoản < 35% OI trung bình)
mean_oi = (df['callOpenInterest'] + df['putOpenInterest']).mean()
df['totalOI'] = df['callOpenInterest'] + df['putOpenInterest']
air_pocket_strikes = df[df['totalOI'] < 0.35 * mean_oi]['strike'].tolist()

# QUY ĐỔI SANG SPOT XAUUSD (Trừ Basis)
S_Implied_Spot  = S_Implied - BASIS
Call_Wall_Spot  = Call_Wall_Futures - BASIS
Put_Wall_Spot   = Put_Wall_Futures - BASIS
Abs_GEX_Spot    = Abs_GEX_Futures - BASIS
Upper_1s_Spot   = Upper_1s - BASIS
Lower_1s_Spot   = Lower_1s - BASIS
Upper_085s_Spot = Upper_085s - BASIS
Lower_085s_Spot = Lower_085s - BASIS

print('='*70)
print(f'🎯 MA TRẬN ĐỊNH LƯỢNG QUANT CHO {SESSION_NAME}')
print(f'• Trạng thái Gamma: {gex_regime} (Tổng Net GEX: {total_net_gex:+.2f}M $)')
print(f'• Major Call Wall (#1 Call OI): Futures={Call_Wall_Futures:.1f}  ==> Spot={Call_Wall_Spot:.2f}')
print(f'• Major Put Wall (#1 Put OI):  Futures={Put_Wall_Futures:.1f}   ==> Spot={Put_Wall_Spot:.2f}')
print(f'• ATM Synthetic Pivot:         Futures={S_Implied:.2f}         ==> Spot={S_Implied_Spot:.2f}')
print(f'• Upper Band (+1.0σ):          Futures={Upper_1s:.2f}          ==> Spot={Upper_1s_Spot:.2f}')
print(f'• Lower Band (-1.0σ):          Futures={Lower_1s:.2f}          ==> Spot={Lower_1s_Spot:.2f}')
print(f'• Dải bảo thủ (±0.85σ):        Spot [{Lower_085s_Spot:.2f} <---> {Upper_085s_Spot:.2f}]')
print(f'• Vùng chân không (Air Pockets): Strikes {air_pocket_strikes}')
print('='*70)


### 📊 Bước 5: Biểu đồ Định lượng Tương tác (Interactive ATAS/TrueVision Style)
Rê chuột vào các cột để xem chi tiết Strike, Net GEX, và các chốt chặn then chốt.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# 1. Call GEX (Xanh lá)
fig.add_trace(go.Bar(
    x=df['strike'],
    y=df['callGEX'],
    name='Call GEX (+$)',
    marker_color='#00e676',
    opacity=0.85
))

# 2. Put GEX (Đỏ cam)
fig.add_trace(go.Bar(
    x=df['strike'],
    y=df['putGEX'],
    name='Put GEX (-$)',
    marker_color='#ff1744',
    opacity=0.85
))

# 3. Đường Net GEX
fig.add_trace(go.Scatter(
    x=df['strike'],
    y=df['netGEX'],
    name='Net GEX Profile',
    line=dict(color='#ffd600', width=3),
    mode='lines+markers'
))

# Vẽ các đường cản chính
fig.add_vline(x=Call_Wall_Futures, line_width=2, line_dash='dash', line_color='#00e676',
              annotation_text=f'Call Wall: {Call_Wall_Futures:.0f} (Spot {Call_Wall_Spot:.1f})', annotation_position='top right')

fig.add_vline(x=Put_Wall_Futures, line_width=2, line_dash='dash', line_color='#ff1744',
              annotation_text=f'Put Wall: {Put_Wall_Futures:.0f} (Spot {Put_Wall_Spot:.1f})', annotation_position='top left')

fig.add_vline(x=S_Implied, line_width=2, line_color='#ffd600',
              annotation_text=f'ATM Pivot: {S_Implied:.1f} (Spot {S_Implied_Spot:.1f})', annotation_position='bottom right')

# Đánh dấu Air Pocket
if air_pocket_strikes:
    min_ap = min(air_pocket_strikes) - 5
    max_ap = max(air_pocket_strikes) + 5
    fig.add_vrect(x0=min_ap, x1=max_ap, fillcolor='rgba(255, 255, 255, 0.08)', line_width=0,
                  annotation_text='AIR POCKET (CHÂN KHÔNG)', annotation_position='top left')

fig.update_layout(
    title=f'<b>NEIRO-QUANT // GEX PROFILE & DEALER POSITIONING ({SESSION_NAME})</b>',
    template='plotly_dark',
    paper_bgcolor='#0d1117',
    plot_bgcolor='#161b22',
    xaxis=dict(title='Strike Price (CME Futures)', showgrid=True, gridcolor='#30363d'),
    yaxis=dict(title='GEX Exposure (Million $)', showgrid=True, gridcolor='#30363d'),
    barmode='relative',
    height=550,
    hovermode='x unified'
)

fig.show()


### 🤖 Bước 6: Trợ lý AI DeepSeek Nhận định Chiến lược Phiên (AI Quant Advisor)
DeepSeek sẽ tự động đọc toàn bộ ma trận số liệu và phân tích: Mức cản nào trụ vững? Mức nào có nguy cơ bị đâm thủng? Điểm Entry, SL, TP cụ thể cho Spot XAUUSD.

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url='https://api.deepseek.com'
)

quant_summary = f'''
PHIÊN GIAO DỊCH: {SESSION_NAME}
Giá CME Futures hiện tại: {FUTURES_PRICE}
Giá XAUUSD Spot sàn giao dịch: {SPOT_PRICE}
Độ lệch Basis (Futures - Spot): {BASIS:+.2f} USD

KẾT QUẢ TÍNH TOÁN QUANT:
- Tổng Net GEX: {total_net_gex:+.2f} Million USD ({gex_regime})
- Major Call Wall: Futures {Call_Wall_Futures:.1f} ==> Spot XAUUSD {Call_Wall_Spot:.2f} (OI: {call_wall_row['callOpenInterest']})
- Major Put Wall:  Futures {Put_Wall_Futures:.1f} ==> Spot XAUUSD {Put_Wall_Spot:.2f} (OI: {put_wall_row['putOpenInterest']})
- ATM Synthetic Pivot: Futures {S_Implied:.2f} ==> Spot XAUUSD {S_Implied_Spot:.2f}
- Dải 1-Sigma (+1σ / -1σ): Spot [{Lower_1s_Spot:.2f} <---> {Upper_1s_Spot:.2f}]
- Dải bảo thủ (±0.85σ): Spot [{Lower_085s_Spot:.2f} <---> {Upper_085s_Spot:.2f}]
- Vùng chân không thanh khoản (Air Pockets): Các Strike CME {air_pocket_strikes}
'''

system_prompt = '''Bạn là Kỹ sư Định lượng & Chuyên gia Cấu trúc Vi mô Thị trường (Market Microstructure Specialist) của hệ thống Neiro-Quant.
Nhiệm vụ của bạn là đọc báo cáo ma trận Options/GEX và xuất ra bản Kế hoạch tác chiến trước phiên cực kỳ sắc sảo, dứt khoát:
1. Đánh giá trạng thái thị trường: Regime hiện tại là gì? Nhà tạo lập (Dealer) đang ghìm vol hay phóng dao trượt giá?
2. Phân tích chốt chặn: Mức nào có khả năng TRỤ VỮNG (Absorption)? Mức nào CÓ NGUY CƠ BỊ ĐÂM THỦNG VÀ TRƯỢT DÀI (Air Pocket)?
3. Lệnh cấm tối cao: Cấm bắt đáy / cấm cản tàu ở vùng nào?
4. Kế hoạch vào lệnh chi tiết cho XAUUSD SPOT: Kèo Đảo chiều (Fade) hoặc Kèo Phá vỡ (Breakout Runner) kèm ENTRY, STOP LOSS VÀ TAKE PROFIT chuẩn xác từng số thập phân.'''

print('⏳ Đang kết nối DeepSeek API để phân tích cấu trúc vi mô...')

try:
    response = client.chat.completions.create(
        model='deepseek-chat',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f'Hãy phân tích dữ liệu Quant sau và đưa ra chiến lược tác chiến:\n{quant_summary}'}
        ],
        temperature=0.3
    )
    
    print('\n' + '='*75)
    print(f'🤖 BẢN KẾ HOẠCH TÁC CHIẾN TỪ DEEPSEEK QUANT ({SESSION_NAME})')
    print('='*75 + '\n')
    print(response.choices[0].message.content)
except Exception as e:
    print('Lỗi gọi DeepSeek API:', e)
